# S26 오라클 진단과 정답 문서를 뺀 거부 세트

검색 기준(S25 베이스라인, 판정용 330질의)과 근거만 바꾼 두 조건의 judge 결과를 나란히 본다. 오라클은 정답 문서가 상위 5개에 없던 질의만, 거부 세트는 정답 문서가 있던 질의만 새로 생성했다(`docs/slices/26-oracle.md`). 판단 재료만 두고 결정(최소 개선 폭, δ_r)은 적지 않는다.

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from rag.gen_eval.checks import HANDLINGS, STATES, unlisted_refusal
from rag.gen_eval.common import read_jsonl
from rag.gen_eval.report import criterion, item_summary
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
CONDITIONS = ["retrieved", "oracle", "no_gold"]
CFGS = {c: load_cfg(["index.scope=full", "+exp=exp017", f"gen_eval.contexts={c}"]) for c in CONDITIONS}
JUDGE = {c: ROOT / CFGS[c].paths.gen_judge for c in CONDITIONS}
# 베이스라인 judge 결과는 S25 실행에서 질의 수를 붙여 저장했다(D-51)
JUDGE["retrieved"] = JUDGE["retrieved"].with_name(JUDGE["retrieved"].stem + "-n330.jsonl")
RUNS = {c: ROOT / CFGS[c].paths.gen_runs for c in CONDITIONS}
KEYS = {"faithfulness": "충실성", "relevance": "답변 관련성", "inclusion": "핵심 답변 포함도"}

In [3]:
def by_qid(path):
    return {r["qid"]: r for r in read_jsonl(path)}


def score(row, key):
    c = row.get(key)
    return None if c is None else c.get("score")

## 데이터 로딩

In [4]:
judged = {c: by_qid(JUDGE[c]) for c in CONDITIONS}
answers = {c: {q: r["answer"] for q, r in by_qid(RUNS[c]).items()} for c in CONDITIONS}
base = judged["retrieved"]
{c: len(judged[c]) for c in CONDITIONS}

{'retrieved': 330, 'oracle': 11, 'no_gold': 319}

In [5]:
# 새로 생성한 질의가 베이스라인 판정용 330 안에 있고, 근거 조건에 맞는지 본다
gold_missing = [q for q, r in base.items() if r["gold_n"] is None]
gold_present = [q for q, r in base.items() if r["gold_n"] is not None]
print("베이스라인 정답 문서 없음", len(gold_missing), "있음", len(gold_present))
print("오라클 질의 = 정답 없던 질의:", set(judged["oracle"]) == set(gold_missing))
print("거부 세트 질의 = 정답 있던 질의:", set(judged["no_gold"]) == set(gold_present))
print("오라클 정답 위치:", Counter(r["gold_n"] for r in judged["oracle"].values()))
print("거부 세트 정답 위치:", Counter(r["gold_n"] for r in judged["no_gold"].values()))

베이스라인 정답 문서 없음 11 있음 319
오라클 질의 = 정답 없던 질의: True
거부 세트 질의 = 정답 있던 질의: True
오라클 정답 위치: Counter({5: 11})
거부 세트 정답 위치: Counter({None: 319})


## 오라클: 정답 문서를 넣은 질의

In [6]:
rows = []
for q in gold_missing:
    b, o = base[q], judged["oracle"][q]
    rows.append({"qid": q, "거부(기준)": b["refusal"], "거부(오라클)": o["refusal"],
                 **{f"{v}(기준)": score(b, k) for k, v in KEYS.items()},
                 **{f"{v}(오라클)": score(o, k) for k, v in KEYS.items()}})
oracle_df = pd.DataFrame(rows).set_index("qid")
oracle_df

,거부(기준),거부(오라클),충실성(기준),답변 관련성(기준),핵심 답변 포함도(기준),충실성(오라클),답변 관련성(오라클),핵심 답변 포함도(오라클)
qid,,,,,,,,
046280-0008,None,None,0.500000,1.0,None,0.250000,1.0,0.0
049848-0012,None,None,1.000000,1.0,None,0.500000,0.5,NaN
043344-0001,None,None,1.000000,0.5,None,1.000000,1.0,NaN
050738-0012,None,None,1.000000,1.0,None,1.000000,1.0,0.0
047535-0009,None,None,1.000000,1.0,None,1.000000,1.0,1.0
052412-0003,None,None,1.000000,1.0,None,1.000000,1.0,0.0
051405-0013,None,None,0.800000,1.0,None,0.800000,1.0,NaN
051405-0025,None,None,0.142857,1.0,None,0.000000,1.0,NaN
046687-0013,None,None,0.750000,1.0,None,0.750000,1.0,NaN


In [7]:
# 오라클 조건 전체 = 정답이 있던 319질의는 베이스라인 결과, 없던 11질의는 오라클 결과
oracle_all = {**base, **judged["oracle"]}
summary = []
for k, v in KEYS.items():
    for name, rows_ in [("검색 기준", list(base.values())), ("오라클", list(oracle_all.values()))]:
        s, ex, err = criterion(rows_, k)
        summary.append({"기준": v, "조건": name, "평균": round(float(np.mean(s)), 3), "계산한 질의": len(s),
                        "뺀 질의": dict(ex), "형식 오류": err})
pd.DataFrame(summary)

,기준,조건,평균,계산한 질의,뺀 질의,형식 오류
0,충실성,검색 기준,0.803,314,"{'full_refusal': 9, 'no_statements': 5}",2
1,충실성,오라클,0.800,315,"{'full_refusal': 9, 'no_statements': 4}",2
2,답변 관련성,검색 기준,0.871,321,{'full_refusal': 9},0
3,답변 관련성,오라클,0.871,321,{'full_refusal': 9},0
4,핵심 답변 포함도,검색 기준,0.512,258,"{'gold_missing': 11, 'full_refusal': 9, 'refer...",1
5,핵심 답변 포함도,오라클,0.510,263,"{'full_refusal': 9, 'reference_unusable': 57}",1


## 검색 실패와 생성 실패

In [8]:
# 정답 문서가 근거에 있는데도 점수가 낮은 질의는 생성(또는 평가) 쪽, 정답 문서가 없던 질의는 검색 쪽이다
fail = []
for k, v in KEYS.items():
    present = [score(base[q], k) for q in gold_present if score(base[q], k) is not None]
    fail.append({"기준": v, "정답 있음 질의 수": len(present), "평균": round(float(np.mean(present)), 3),
                 "1 미만 질의": sum(s < 1 for s in present), "0.5 미만 질의": sum(s < 0.5 for s in present),
                 "정답 없음 질의 수(검색 실패)": len(gold_missing)})
pd.DataFrame(fail)

,기준,정답 있음 질의 수,평균,1 미만 질의,0.5 미만 질의,정답 없음 질의 수(검색 실패)
0,충실성,304,0.803,103,43,11
1,답변 관련성,310,0.868,68,14,11
2,핵심 답변 포함도,258,0.512,168,109,11


## 거부 세트: 정답 문서를 뺀 질의

In [9]:
ng = judged["no_gold"]
kinds = Counter(r["refusal"] or "거부 아님" for r in ng.values())
# 지정 문구 거부(코드 판별)와 지정 문구 없는 거부(PR #158 규칙)를 따로 센다
phrase = CFGS["no_gold"].prompt.refusal
parts = list(CFGS["no_gold"].gen_eval.unlisted_refusal)
unlisted = sum(unlisted_refusal(answers["no_gold"][q], phrase, parts) for q in ng)
base_kinds = Counter(base[q]["refusal"] or "거부 아님" for q in gold_present)
pd.DataFrame({"검색 기준(같은 질의)": base_kinds, "거부 세트": kinds}).fillna(0).astype(int), \
    f"거부 세트 중 지정 문구 없는 거부 {unlisted}개"

(         검색 기준(같은 질의)  거부 세트
 거부 아님             309    173
 full                9    144
 partial             1      2,
 '거부 세트 중 지정 문구 없는 거부 58개')

In [10]:
s = item_summary(list(ng.values()))
table = pd.DataFrame([[s["table"].get((st, h), 0) for h in HANDLINGS] for st in STATES],
                     index=STATES, columns=HANDLINGS)
print("질의", s["queries"], "형식 오류", s["errors"], "거부 정확도(참고값)", s["accuracy"])
table

질의 319 형식 오류 0 거부 정확도(참고값) (290, 436)


,답함,부분임을 밝히고 답함,상황을 밝히고 답함,거부,다루지 않음
있음,83,6,5,20,1
일부,31,6,6,26,5
없음,22,4,5,196,20
모호,8,2,4,10,4
회피,3,0,1,4,3


In [11]:
# 정답 문서를 뺐는데 정보 '있음'으로 판정된 항목: 다른 문서에 답이 있거나 judge가 잘못 본 경우(라벨 잡음)
items = [i for r in ng.values() for i in (r.get("item_eval") or {}).get("items") or []]
states = Counter(i["state"] for i in items)
print("항목", len(items), dict(states))
print("판정", dict(Counter(i["outcome"] for i in items)))

항목 475 {'일부': 74, '있음': 115, '회피': 11, '없음': 247, '모호': 28}
판정 {'과잉 거부': 46, '맞음': 290, '상황 미전달': 34, '안내 없이 부분 답함': 37, '근거 없이 답함': 31, '불필요한 부분 안내': 6, '무언 생략': 20, '상황 전달': 5, '누락': 6}


In [12]:
# 거부하지 않은 답변의 충실성: 정답 문서 없이 답한 답변이 근거 밖 내용을 얼마나 담는가
answered = [r for r in ng.values() if r["refusal"] != "full"]
for k, v in KEYS.items():
    sc, ex, err = criterion(answered, k)
    print(v, "평균", round(float(np.mean(sc)), 3) if sc else "-", "질의", len(sc), "뺀", dict(ex), "형식 오류", err)

충실성 평균 0.792 질의 164 뺀 {'no_statements': 11} 형식 오류 0
답변 관련성 평균 0.789 질의 175 뺀 {} 형식 오류 0
핵심 답변 포함도 평균 - 질의 0 뺀 {'gold_missing': 175} 형식 오류 0


## 관찰

- 검색 실패의 몫이 작다. 판정용 330질의 중 정답 문서가 근거에 없던 질의는 11개이고, 이 11개에 정답 문서를 넣어도 세 기준 평균은 거의 같다(충실성 0.803 → 0.800, 관련성 0.871 → 0.871, 포함도 0.512 → 0.510, 포함도 계산 질의 258 → 263)
- 정답 문서가 근거에 있는 질의에서도 포함도 1 미만이 168/258, 0.5 미만이 109/258이다. 점수가 낮은 원인은 대부분 생성(또는 평가) 쪽에 있다
- 정답 문서를 빼면 319질의 중 전체 거부 144, 부분 거부 2, 거부 아님 173이다. 거부 아님 중에는 지정 문구 없는 거부가 있다(58개, PR #158 규칙)
- 정답 문서를 뺀 근거에서도 judge는 475항목 중 115항목(24%)을 정보 '있음'으로 판정했다. 다른 문서에 답이 있거나 judge가 잘못 본 경우로, 거부 세트 라벨의 잡음 크기다
- 정보 '없음' 247항목의 처리는 거부 196, 다루지 않음 20, 답함 22 등이고, 판정 '근거 없이 답함'은 31항목이다
- 거부하지 않은 답변의 충실성 평균은 0.792(164질의)로 검색 기준(0.803)과 비슷하다
